# 基于 Transformer 的 Keyword Spotting

In [2]:
!pip install -q transformers datasets soundfile librosa

In [3]:
from datasets import load_dataset
from transformers import pipeline
from IPython.display import Audio, display

我们将使用 Hugging Face 生态构建一个端到端的音频分类器。目标是：输入一段真实的银行客服录音，让大模型自动判断客户的来电意图（比如：付款、挂失、查询等）

### 第 1 步：拉取数据 (Load Dataset)

In [4]:
# ==========================================
# 模块：Keyword Spotting (关键词/意图识别)
# 目标：听取一段银行客服录音，精准分类用户的来电意图
# ==========================================


# 我们使用的是澳洲英语 (en-AU) 子集，包含真实的银行客服呼叫录音
minds = load_dataset("PolyAI/minds14", name="en-AU", split="train")
raw_audio = minds[1]["audio"]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

en-AU/train-00000-of-00001.parquet:   0%|          | 0.00/37.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/654 [00:00<?, ? examples/s]

### 步骤二：实例化 AI 流水线 (Pipeline)
使用 Hugging Face 的 pipeline 接口，只需两行代码即可完成极其复杂的 Transformer 架构加载和特征提取器的初始化。

In [5]:
classifier = pipeline(
    task="audio-classification",
    model="anton-l/xtreme_s_xlsr_300m_minds14"
)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

### 步骤三：数据维度清洗
Hugging Face datasets 加载的音频偶尔会携带一个表示声道数量的多余维度（例如形状为 (1, N)）。我们需要使用 .squeeze() 方法将其降维成纯粹的一维数组 (N,)


In [6]:
clean_audio_input = {
    "array": raw_audio["array"].squeeze(),
    "sampling_rate": raw_audio["sampling_rate"]
}

In [7]:
predictions = classifier(clean_audio_input)

for p in predictions:
  print(f"label:{p['label']:<20} | acc:{p['score']:.4%}")

display(Audio(clean_audio_input["array"], rate=clean_audio_input["sampling_rate"]))

label:pay_bill             | acc:98.4216%
label:high_value_payment   | acc:0.4519%
label:direct_debit         | acc:0.3372%
label:freeze               | acc:0.2500%
label:card_issues          | acc:0.2091%
label:latest_transactions  | acc:0.0697%
label:abroad               | acc:0.0619%
label:address              | acc:0.0492%
label:atm_limit            | acc:0.0377%
label:balance              | acc:0.0305%
label:app_error            | acc:0.0277%
label:joint_account        | acc:0.0229%
label:business_loan        | acc:0.0192%
label:cash_deposit         | acc:0.0114%


# Language Identification
在这个模块中，将测试模型分辨不同语言的能力。
使用 FLEURS 数据集（包含 102 种语言），并调用一个由强大语音识别模型 Whisper 修改而成的语种分类器

In [9]:
from datasets import load_dataset

lid_classifier = pipeline(
    task="audio-classification",
    model="sanchit-gandhi/whisper-medium-fleurs-lang-id"
)

languages_to_test = ["fr-FR", "zh-CN", "de-DE", "es-ES"]

for lang in languages_to_test:
  dataset = load_dataset("PolyAI/minds14", name=lang, split="train", streaming=True)
  print(f"dataset of {lang} has loaded. ")

  sample = next(iter(dataset))
  clean_input = {
      "array": sample["audio"]["array"].squeeze(),
      "sampling_rate": sample["audio"]["sampling_rate"]
  }

  print(f"sample of {lang} has loaded. ")
  print(f"start to predict.")
  all_predictions = lid_classifier(clean_input)
  for p in all_predictions:
    print(f"label:{p['label']:<20} | acc:{p['score']:.4%}")

  display(Audio(clean_input["array"], rate=clean_input["sampling_rate"]))


Loading weights:   0%|          | 0/371 [00:00<?, ?it/s]

dataset of fr-FR has loaded. 
sample of fr-FR has loaded. 
start to predict.
label:French               | acc:99.9512%
label:Occitan              | acc:0.0148%
label:Hebrew               | acc:0.0048%
label:Asturian             | acc:0.0030%
label:Georgian             | acc:0.0021%
label:Wolof                | acc:0.0008%
label:Maori                | acc:0.0008%
label:Pashto               | acc:0.0007%
label:Mongolian            | acc:0.0006%
label:Armenian             | acc:0.0005%
label:Assamese             | acc:0.0004%
label:Czech                | acc:0.0004%
label:Vietnamese           | acc:0.0004%
label:Korean               | acc:0.0003%
label:Lingala              | acc:0.0003%
label:Bosnian              | acc:0.0003%
label:Yoruba               | acc:0.0003%
label:Japanese             | acc:0.0003%
label:Slovenian            | acc:0.0002%
label:Catalan              | acc:0.0002%
label:Arabic               | acc:0.0002%
label:Amharic              | acc:0.0002%
label:Russian       

dataset of zh-CN has loaded. 
sample of zh-CN has loaded. 
start to predict.
label:Mandarin Chinese     | acc:100.0000%
label:Cantonese Chinese    | acc:0.0002%
label:Mongolian            | acc:0.0002%
label:Urdu                 | acc:0.0001%
label:Spanish              | acc:0.0001%
label:Bengali              | acc:0.0001%
label:Bosnian              | acc:0.0001%
label:Irish                | acc:0.0001%
label:Swedish              | acc:0.0001%
label:Japanese             | acc:0.0001%
label:Maori                | acc:0.0001%
label:French               | acc:0.0001%
label:Georgian             | acc:0.0001%
label:Pashto               | acc:0.0001%
label:Persian              | acc:0.0001%
label:Oromo                | acc:0.0000%
label:Burmese              | acc:0.0000%
label:Dutch                | acc:0.0000%
label:Italian              | acc:0.0000%
label:Yoruba               | acc:0.0000%
label:Lao                  | acc:0.0000%
label:Kyrgyz               | acc:0.0000%
label:Azerbaijani  

dataset of de-DE has loaded. 
sample of de-DE has loaded. 
start to predict.
label:Icelandic            | acc:24.1699%
label:Welsh                | acc:22.8760%
label:Armenian             | acc:20.8374%
label:Dutch                | acc:11.6394%
label:Azerbaijani          | acc:3.6774%
label:Catalan              | acc:2.2964%
label:Luxembourgish        | acc:1.8814%
label:Asturian             | acc:1.4915%
label:Norwegian            | acc:1.0368%
label:Irish                | acc:0.9918%
label:Arabic               | acc:0.6828%
label:Georgian             | acc:0.6317%
label:Lithuanian           | acc:0.3887%
label:Amharic              | acc:0.3716%
label:Swedish              | acc:0.3542%
label:Estonian             | acc:0.3414%
label:Yoruba               | acc:0.3054%
label:Mandarin Chinese     | acc:0.3004%
label:Slovenian            | acc:0.2968%
label:Bulgarian            | acc:0.2817%
label:Galician             | acc:0.2621%
label:Thai                 | acc:0.2542%
label:Kyrgyz     

dataset of es-ES has loaded. 
sample of es-ES has loaded. 
start to predict.
label:Asturian             | acc:100.0000%
label:Galician             | acc:0.0039%
label:Catalan              | acc:0.0005%
label:French               | acc:0.0003%
label:Bosnian              | acc:0.0003%
label:Greek                | acc:0.0003%
label:Maori                | acc:0.0003%
label:Japanese             | acc:0.0002%
label:Dutch                | acc:0.0001%
label:Thai                 | acc:0.0001%
label:Yoruba               | acc:0.0001%
label:Mandarin Chinese     | acc:0.0001%
label:Czech                | acc:0.0001%
label:Zulu                 | acc:0.0001%
label:Sorani-Kurdish       | acc:0.0001%
label:Korean               | acc:0.0001%
label:Georgian             | acc:0.0001%
label:Northern-Sotho       | acc:0.0001%
label:Mongolian            | acc:0.0001%
label:Danish               | acc:0.0001%
label:Slovak               | acc:0.0001%
label:Burmese              | acc:0.0001%
label:Amharic      